In [1]:
#From new, calculate pressure at 1 bar, then go back to before import logging
#git clone -b djbowerpreview2
import numpy as np
import optimistix as optx
from atmodeller import (
    EquilibriumModel,
    Planet,
    ChemicalSpecies,
    SpeciesNetwork,
    debug_logger,
    earth_oceans_to_hydrogen_mass,
    SolverParameters,
)
from atmodeller.eos import get_eos_models
from atmodeller.solubility import get_solubility_models
from atmodeller.thermodata import IronWustiteBuffer
from atmodeller.interfaces import FugacityConstraintProtocol, ThermodynamicStateProtocol
from atmodeller import debug_logger
from atmodeller.classes import EquilibriumModel
from atmodeller.containers import (
    ChemicalSpecies,
    FixedFugacityConstraint,
    SpeciesNetwork,
    ThermodynamicState,
)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import LogNorm
from matplotlib.patches import Patch

from scipy.interpolate import interp1d
from scipy.optimize import fsolve, root_scalar
import os

from matplotlib.ticker import ScalarFormatter

formatter = ScalarFormatter(useMathText=False)
formatter.set_scientific(False)
formatter.set_useOffset(False)
from mpl_toolkits.axes_grid1.axes_divider import make_axes_locatable

logger = debug_logger() #you can dump it in file 

#gamma 0.05 0.01 guillot 2010 f 1/4 tau compute 

Atmodeller initialized with double precision (float64)


In [3]:
color_tot = "black"
color_H2 = "orange"
color_H2O = "blue"
color_SiH4 = "red"
color_SiO2 = "yellow"
color_SiO = "brown"
color_O2 = "limegreen"
color_CO2 = "cyan"
color_CO = "magenta"
color_CH4 = "purple"
color_N2 = "green"
color_NH3 = "pink"
color_MgO = "darkred"
color_He = "gray"
color_H = "orange"
color_C = "cyan"
color_N = "green"
color_O = "blue"
color_Si = "red"
color_Mg = "lime"
color_MgH2 = "gold"
color_MgH = "teal"
color_MgOH = "indigo"
color_Ccr = "dimgray"

In [2]:
SiO2_mantlemasspercent_palme14: float = 45.4
total_mantlemasspercent_palme14: float = 98.41
core_mass_fraction: float = 0.327  # typical values used are between 0.325 - 0.33
Si_massfraction_palme14: float = round(
    SiO2_mantlemasspercent_palme14
    / total_mantlemasspercent_palme14
    * (1 - core_mass_fraction)
    / (28.0855 + 2 * 15.999)
    * 28.0855,
    3,
)

O_massfraction_palme14: float = Si_massfraction_palme14 / 28.0855 * 2 * 15.999
# Lodders et al. (2009) Springer Book Chapter - Table 8 (in wt%)

H_masspercent_lodders09: float = 73.9
He_masspercent_lodders09: float = 24.69
C_masspercent_lodders09: float = 0.22
N_masspercent_lodders09: float = 0.07
O_masspercent_lodders09: float = 0.63
Si_masspercent_lodders09: float = 0.07
Mg_masspercent_lodders09: float = 0.06

# Lodders et al. (2009) Springer Book Chapter - Table 6 (log-normalalized abundances relative to H)

H_logN: float = 12
He_logN: float = 10.93
C_logN: float = 8.39
N_logN: float = 7.86
O_logN: float = 8.73
Si_logN: float = 7.53
Mg_logN: float = 7.54 

In [4]:
eos_models = get_eos_models()
sol_models = get_solubility_models()

H2O_g = ChemicalSpecies.create_gas("H2O")
H2O_gs = ChemicalSpecies.create_gas("H2O", solubility=sol_models["H2O_peridotite_sossi23"])
H2O_rgs = ChemicalSpecies.create_gas(
    "H2O",
    activity=eos_models["H2O_cork_holland98"],
    solubility=sol_models["H2O_peridotite_sossi23"],
)
H2O_rg = ChemicalSpecies.create_gas(
    "H2O",
    activity=eos_models["H2O_cork_holland98"],
)

H2_g = ChemicalSpecies.create_gas("H2")
H2_gs = ChemicalSpecies.create_gas("H2", solubility=sol_models["H2_basalt_hirschmann12"])
H2_rgs = ChemicalSpecies.create_gas(
    "H2", activity=eos_models["H2_chabrier21"], solubility=sol_models["H2_basalt_hirschmann12"]
)
H2_rg = ChemicalSpecies.create_gas(
    "H2", activity=eos_models["H2_chabrier21"]
)

O2_g = ChemicalSpecies.create_gas("O2")
O2_rg = ChemicalSpecies.create_gas("O2", activity=eos_models["O2_cs_shi92"])

OSi_g = ChemicalSpecies.create_gas("OSi")
OSi_rg = ChemicalSpecies.create_gas("OSi", activity=eos_models["OSi_rk49_connolly16"])

H4Si_g = ChemicalSpecies.create_gas("H4Si")
H4Si_rg = ChemicalSpecies.create_gas("H4Si", activity=eos_models["H4Si_wang18"])

O2Si_cd = ChemicalSpecies.create_condensed("O2Si", state="cd")

C_cd = ChemicalSpecies.create_condensed("C", state="cd")
CSi_cd = ChemicalSpecies.create_condensed("CSi", state="cd")
Si_cd = ChemicalSpecies.create_condensed("Si", state="cd")
N4Si3_cd = ChemicalSpecies.create_condensed("N4Si3", state="cd")
H2O_cd = ChemicalSpecies.create_condensed("H2O", state="cd")

CO2_g = ChemicalSpecies.create_gas("CO2")
CO2_gs = ChemicalSpecies.create_gas("CO2", solubility=sol_models["CO2_basalt_dixon95"])
CO2_rgs = ChemicalSpecies.create_gas(
    "CO2", solubility=sol_models["CO2_basalt_dixon95"], activity=eos_models["CO2_cs_shi92"]
)
CO2_rg = ChemicalSpecies.create_gas(
    "CO2",activity=eos_models["CO2_cs_shi92"]
)

CO_g = ChemicalSpecies.create_gas("CO")
CO_gs = ChemicalSpecies.create_gas("CO", solubility=sol_models["CO_basalt_yoshioka19"])
CO_rgs = ChemicalSpecies.create_gas(
    "CO", solubility=sol_models["CO_basalt_yoshioka19"], activity=eos_models["CO_cs_shi92"]
)
CO_rg = ChemicalSpecies.create_gas(
    "CO", activity=eos_models["CO_cs_shi92"]
)

CH4_g = ChemicalSpecies.create_gas("CH4")
CH4_gs = ChemicalSpecies.create_gas("CH4", solubility=sol_models["CH4_basalt_ardia13"])
CH4_rgs = ChemicalSpecies.create_gas(
    "CH4", solubility=sol_models["CH4_basalt_ardia13"], activity=eos_models["CH4_cs_shi92"]
)
CH4_rg = ChemicalSpecies.create_gas(
    "CH4", activity=eos_models["CH4_cs_shi92"]
)

N2_g = ChemicalSpecies.create_gas("N2")
N2_gs = ChemicalSpecies.create_gas("N2", solubility=sol_models["N2_basalt_libourel03"])
N2_rgs = ChemicalSpecies.create_gas(
    "N2",solubility=sol_models["N2_basalt_libourel03"], activity=eos_models["N2_cs_saxena87"]
)
N2_rg = ChemicalSpecies.create_gas(
    "N2", activity=eos_models["N2_cs_saxena87"]
)


NH3_g = ChemicalSpecies.create_gas("H3N")
NH3_rg = ChemicalSpecies.create_gas("H3N", activity=eos_models["H3N_rk49_reid87"])

He_g = ChemicalSpecies.create_gas("He")
He_gs = ChemicalSpecies.create_gas("He", solubility=sol_models["He_basalt_jambon86"])
He_rgs = ChemicalSpecies.create_gas(
    "He", solubility=sol_models["He_basalt_jambon86"], activity=eos_models["He_chabrier21"]
)
He_rg = ChemicalSpecies.create_gas(
    "He", activity=eos_models["He_chabrier21"]
)


species_HOSi_magma_nosol_ideal = SpeciesNetwork((H2O_g, H2_g, O2_g, OSi_g, H4Si_g, O2Si_cd))
species_HOSi_magma_sol_ideal = SpeciesNetwork((H2O_gs, H2_gs, O2_g, OSi_g, H4Si_g, O2Si_cd))
species_HOSi_magma_sol_real = SpeciesNetwork((H2O_rgs, H2_rgs, O2_rg, OSi_rg, H4Si_rg, O2Si_cd))
species_HOSi_atm_sol_real = SpeciesNetwork((H2O_g, H2_g, O2_g, H2O_cd,OSi_rg, H4Si_rg))


species_HHeCNOSi_magma_nosol_ideal = SpeciesNetwork(
    (H2O_g, H2_g, O2_g, OSi_g, H4Si_g, O2Si_cd, CO2_g, CO_g, CH4_g, N2_g, NH3_g, He_g)
)
species_HHeCNOSi_magma_sol_ideal = SpeciesNetwork(
    (H2O_gs, H2_gs, O2_g, OSi_g, H4Si_g, O2Si_cd, CO2_gs, CO_gs, CH4_gs, N2_gs, NH3_g, He_gs)
)
species_HHeCNOSi_magma_sol_real = SpeciesNetwork(
    (
        H2O_rgs,
        H2_rgs,
        O2_rg,
        OSi_rg,
        H4Si_rg,
        O2Si_cd,
        CO2_rgs,
        CO_rgs,
        CH4_rgs,
        N2_rgs,
        NH3_rg,
        He_rgs,
    )
)

species_HHeCNOSi_atmosphere = SpeciesNetwork(
    (H2O_rg, H2_rg, O2_rg, OSi_rg, H4Si_rg, H2O_cd, C_cd, CSi_cd, Si_cd, N4Si3_cd,
     CO2_rg, CO_rg, CH4_rg, N2_rg, NH3_rg, He_rgs)
)
species_HHeCNOSi_atmosphere = SpeciesNetwork(
    (H2O_g, H2_g, O2_g, OSi_g, H4Si_g, O2Si_cd, C_cd, CSi_cd, Si_cd, N4Si3_cd,
     CO2_g, CO_g, CH4_g, N2_g, NH3_g, He_g)
)

In [5]:
number_of_realisations = 50

# Overall structure 
R_earth = 6.371e6
R_uranus = 3.939 * R_earth
M_earth = 5.972e24 #kg
M_uranus = 14.536 * M_earth

#fO2_log10_shifts = [-1]
fO2_log10_shifts = np.linspace(-6, 0, num=30)  # IW-6 to IW
fO2_log10_shifts = [-6,-5,-4,-3,-2,-1,0]
hmps = np.logspace(1.2,1.2, num=1)  # wt% H
h_kgs = hmps / 100 * M_uranus  # kg

si_kgs_solar = h_kgs * Si_masspercent_lodders09 / H_masspercent_lodders09
mg_kgs_solar = h_kgs * Mg_masspercent_lodders09 / H_masspercent_lodders09
o_kgs_solar = h_kgs * O_masspercent_lodders09 / H_masspercent_lodders09
c_kgs_solar = h_kgs * C_masspercent_lodders09 / H_masspercent_lodders09
n_kgs_solar = h_kgs * N_masspercent_lodders09 / H_masspercent_lodders09
he_kgs_solar = h_kgs * He_masspercent_lodders09 / H_masspercent_lodders09


rho_centres = [2500,3000,2000,1500,1000]
rho_centres = [3000]
rho_centre = 3000
print(rho_centres)
R_core = 0.7*R_uranus
Lowerbound_M_core = [
    4*np.pi*(R_core**3)*rho/3
    for rho in rho_centres
]
M_cores = Lowerbound_M_core
print(M_cores)

[3000]
[6.812158460855618e+25]


In [6]:
f = 1/4

T_solar = 5772
R_solar = 6.957e8
d_solar_Uranus = 2876679082000

T_irr = T_solar * np.sqrt(R_solar/d_solar_Uranus)

T_eff = 59.1 #pm 0.3 The albedo, effective temperature, and energy balance of Uranus, as determined from Voyager IRIS data
T_eq = 58.2 #pm 1 The albedo, effective temperature, and energy balance of Uranus, as determined from Voyager IRIS data
T_int = (T_eff**(4) - T_eq**(4))**(1/4)
#T_int = 120


gamma_1 = 0.01
gamma_2 = 2
gamma_3 = 0.1
gammas = np.logspace(-4,2,7)
gamma = gamma_2
tau = np.logspace(-10,10,1000)

In [7]:
def Temp_profile(tau, T_int, gamma):
    #This function gives back a Temperature, given optical depth, gamma and intrinsic temperature
    #tau : give an optical depth
    #T_int : give the intrinsic temperature
    #T_irr : taken to be constant along this short evolutionary track
    #gamma : gamme based on opacities of visible on thermal
    return (((3*T_int**(4))/4)*((2/3)+tau) + ((3*f*T_irr**(4))/4)*((2/3)+ (1/(gamma*np.sqrt(3))) + ((gamma/(np.sqrt(3)))- (1/(gamma*np.sqrt(3))))*np.exp(-gamma*tau*np.sqrt(3))))**(1/4)  

def find_tau(tau,T_int, gamma, target, initial_guess):
    #This function gives back the optical depth at which a certain temperature can be found, given several parameters
    #tau: space of optical depths to go over
    #T_int : intrinsic temperature
    #gamma : gives the gamma 
    #target: the target temperature
    #intial_guess : your initial guess for the optical depth
    def g(tau):
        return Temp_profile(tau,T_int, gamma) - target
    x_solution = fsolve(g, initial_guess)
    return x_solution[0]

def find_gamma(tau,T_int, gamma, target, initial_guess):
    #This function gives back the gamma at which a certain temperature can be found, given several parameters
    #tau: optical depth at which it is found
    #T_int : intrinsic temperature
    #gamma : gammas to loop over
    #target: the target temperature
    #intial_guess : your initial guess for the optical depth
    def h(gamma):
        return Temp_profile(tau,T_int, gamma) - target
    x_solution = fsolve(h, initial_guess)
    return x_solution[0]
def find_scale(Temperature,T_int,gamma, Pressure, tau, initial_guess):
    #Probably will not be used in the future, finds how optical depth and Pressure are scaled
    #Temperature: gives the temperature at which to calculate the scaling
    #T_int : intrinsic temperature
    #gamma : gamma
    #Pressure: gives the pressure at which to calculate the scaling 
    #denotes the optical depths to loop over 
    #intiial guess for optical depth
    tau_scale = find_tau(tau,T_int,gamma, Temperature, initial_guess)
    scale_factor = Pressure/tau_scale
    return scale_factor

def find_Tint(tau,T_int, gamma, target, initial_guess):
    #This function gives back the T_intr at which a certain temperature can be found, given several parameters
    #tau: optical depth at which it is found
    #T_int : intrinsic temperature to loop over
    #gamma : gamma
    #target: the target temperature
    #intial_guess : your initial guess for the optical depth
    def ze(T_int):
        return Temp_profile(tau,T_int, gamma) - target
    x_solution = fsolve(ze, initial_guess)
    return x_solution[0]
    

def Pressure_Temperatures(T_original, T_int,gamma, tau, Pressure, initial_guess):
    #Gives Pressures/Temperatures profiles given different parameters
    #T_original: Temperature where we know the pressures
    #T_int : the intrinisic temperatures
    #gamma : gamma 
    #tau: optical depths to find over 
    #Pressure: pressure where we know the temperature
    #intial_guess : initial guess for optical depth
    scales = find_scale(T_original, T_int, Pressure, tau, initial_guess)
    Pressures = tau*scales
    Temperatures = Temp_profile(tau, T_int,gamma)
    indices = [i for i, v in enumerate(Temperatures) if 0 < v < 6000]
    return Pressures[indices], Temperatures[indices]

In [8]:
def peridotite_solidus(P_bar): #On the cooling of a deep terrestrial magma ocean
    #P_bar: pressure in bar
    #Returns: solidus temperature in K
    P_Pa = P_bar*10**5
    T_sol = 1661.2*((P_Pa/(1.336*10**9))+1)**(1/7.437)
    return T_sol

def peridotite_liquidus(P_bar):
    #P_bar: pressure in bar
    #Returns: liquidus temperature in K
    P_Pa = P_bar*10**5
    T_liq = 1982.1*((P_Pa/(6.594*10**9))+1)**(1/5.374)
    
    return T_liq

In [9]:
def opacity(P, T, met): 
    #GASEOUS MEAN OPACITIES FOR GIANT PLANET AND ULTRACOOL DWARF ATMOSPHERES OVER A RANGE OF METALLICITIES AND TEMPERATURES
    #P in bar 
    #T in Kelvin
    #met in [M/H], different then how we define elsewhere
    c1 = 10.602
    c2 = 2.882
    c3 = 6.09e-15
    c4 = 2.954
    c5 = -2.526
    c6 = 0.843
    c7 = -5.490
    
    c8low = -14.051
    c9low = 3.055
    c10low = 0.024
    c11low = 1.877
    c12low = -0.445
    c13low = 0.8321
    
    c8high =  82.241
    c9high = -55.456
    c10high = 8.754 
    c11high = 0.7048
    c12high = -0.0414
    c13high = 0.8321

    if P <= 0 or T <= 0:
        return np.inf
    kappa_low_P = c1*np.arctan(np.log10(T)-c2) - (c3*np.exp((np.log10(T)-c5)**2)/(np.log10(P)+c4)) +c6*met +c7
    if T <= 800:
        kappa_high_P = c8low + c9low*np.log10(T)+c10low*(np.log10(T)**2)+ (c11low + c12low*np.log10(T))*np.log10(P) + c13low*met*(0.5 + (np.arctan(5*np.log10(T)-12.5)/np.pi))
    else:
        kappa_high_P = c8high + c9high*np.log10(T)+c10high*(np.log10(T)**2)+ (c11high + c12high*np.log10(T))*np.log10(P) + c13high*met*(0.5 + (np.arctan(5*np.log10(T)-12.5)/np.pi))
    
    kappa_final = 10**(kappa_low_P) + 10**(kappa_high_P)

    return kappa_final

def optical_depth_finder(P, T, met, surf_grav):
    #Function that gives back optical depths
    #P in cgs
    #T in Kelvin
    #met in [M/H], different then how we define elsewhere
    # the surface gravity
    opt_depth = P*opacity(P, T, met)/surf_grav
    return opt_depth

def optical_depth_finder_given_opacity(P, opacity, surf_grav):
    #Function that gives back optical depths
    #P in cgs
    #T in Kelvin
    #met in [M/H], different then how we define elsewhere
    # the surface gravity
    opt_depth = P*opacity/surf_grav
    return opt_depth

def pressure_finder(optical_depths, T, met, surf_grav, initial_guess):
    #optical depths 
    #T in Kelvin
    #met in [M/H], different then how we define elsewhere
    # the surface gravity
    #gives back pressure in dyne/cm^2
    def az(press):
        return optical_depth_finder(press, T, met, surf_grav) - optical_depths
    x_solution = fsolve(az, initial_guess)
    return x_solution[0]

def pressure_finder_given_opacity(optical_depths, opacity, surf_grav):
    #optical depths 
    #T in Kelvin
    #met in [M/H], different then how we define elsewhere
    # the surface gravity
    #gives back pressure in dyne/cm^2
    pressi = optical_depths*surf_grav/opacity
    return pressi

In [10]:
def Profile_From_1_bar(Temp_at_1bar, grav_at_1bar, metalli, grid_resolution,T_max):
    #Creates a Pressure-Temperature profile from temperatures at 1 bar. 
    #Temp_at_1bar : The temperature at 1 bar level
    #grav_at_1bar : The gravity at the 1 bar level, please in cgs
    #metalli: metallicit in [M/H]
    #grid_resolution : how much space you want 
    #T_max : at some temperatures our later simulations will fail, therefor define a T_max

    Press_1bar_to_cgs = 10**6 #to dyne/cm^2, this is Pressure of 1bar in dyne/cm^2
    
    #Step 1: deduce optical depth at 76K level, the Uranus we know of today 
    optical_depth_76K = optical_depth_finder(Press_1bar_to_cgs, 76, metalli,grav_at_1bar)
#    print(f"Optical depth at 76K in Uranus as of today: {optical_depth_76K}")
    
    #Step 2: deduce an estimation for gamma 
    gammas = np.logspace(-2,0,1000)
    T_eff_final = 59.1 
    T_eq_final = 58.2
    T_int_final = (T_eff_final**(4) - T_eq_final**(4))**(1/4)
    final_gamma = find_gamma(optical_depth_76K,T_int_final, gammas, 76, 0.3)
#    print(f"The gamma to be used is : {final_gamma}")
    
    #Step 3: deduce the optical depth at which our 1bar-level is
    optical_depth_our_case = Press_1bar_to_cgs*opacity(Press_1bar_to_cgs, Temp_at_1bar, metalli)/grav_at_1bar
    thermal_opacity = opacity(Press_1bar_to_cgs, Temp_at_1bar, metalli)
#    print(f"The optical depth at 1 bar in our model: {optical_depth_our_case}")

    #Step 4: deduce which intrinsic Temperature is needed to recover this optical depth at this Pressure and Temperature
    T_int_lin = np.linspace(29,500,1000)
    Tint = find_Tint(optical_depth_our_case,T_int_lin, final_gamma, Temp_at_1bar, 50)
#    print(f"The intrinsic temperature {Tint} in our model")

    #Step 5: Using an upper and lower limit on temperature, find the max optical depth and minimal optical depth
    possible_depths = np.logspace(-10,10,1000)
    final_optical_depth  = find_tau(possible_depths,Tint, final_gamma, T_max, 1e5)
#    print(f"The optical depth until we go {final_optical_depth}")

    #Step 6, over the different optical depths, find all the temperatures
    all_optical_depths = np.logspace(-1,np.log10(final_optical_depth),grid_resolution)
    All_Temperatures = Temp_profile(all_optical_depths, Tint,final_gamma)
#    print(All_Temperatures)
    All_pressures = []
    #Step 7, at different optical depths find different pressures
    for i in range(len(all_optical_depths)):
        aze = pressure_finder_given_opacity(all_optical_depths[i], thermal_opacity, grav_at_1bar)/(10**6)
        #aze = pressure_finder(all_optical_depths[i], All_Temperatures[i], metalli, grav_at_1bar, 1e13)/(10**6)
        All_pressures.append(aze)
#        print(f"At optical depth {all_optical_depths[i]} and Temperature {All_Temperatures[i]}K we have pressures {aze}")

    #Extra Step ~ Find effective temperature 
    T_eq = 58.2 #pm 1 The albedo, effective temperature, and energy balance of Uranus, as determined from Voyager IRIS data
    T_eff_end = (Tint**(4) + T_eq**(4))**(1/4)
    return All_Temperatures, All_pressures, Tint, T_eff_end

In [11]:
def Pressure_at_Temperature_From_1_bar(Temp_at_1bar, grav_at_1bar, metalli, T_forpress):
    #Creates a Pressure-Temperature profile from temperatures at 1 bar and then tries to find the Pressure at a given temperature
    #Temp_at_1bar : The temperature at 1 bar level
    #grav_at_1bar : The gravity at the 1 bar level, please in cgs
    #metalli: metallicit in [M/H]
    #grid_resolution : how much space you want 
    #T_max : at some temperatures our later simulations will fail, therefor define a T_max

    Press_1bar_to_cgs = 10**6 #to dyne/cm^2, this is Pressure of 1bar in dyne/cm^2
    
    #Step 1: deduce optical depth at 76K level, the Uranus we know of today 
    optical_depth_76K = optical_depth_finder(Press_1bar_to_cgs, 76, metalli,grav_at_1bar)
#    print(f"Optical depth at 76K in Uranus as of today: {optical_depth_76K}")
    
    #Step 2: deduce an estimation for gamma 
    gammas = np.logspace(-2,0,1000)
    T_eff_final = 59.1 
    T_eq_final = 58.2
    T_int_final = (T_eff_final**(4) - T_eq_final**(4))**(1/4)
    final_gamma = find_gamma(optical_depth_76K,T_int_final, gammas, 76, 0.3)
#    print(f"The gamma to be used is : {final_gamma}")
    
    #Step 3: deduce the optical depth at which our 1bar-level is
    optical_depth_our_case = Press_1bar_to_cgs*opacity(Press_1bar_to_cgs, Temp_at_1bar, metalli)/grav_at_1bar
    thermal_opacity = opacity(Press_1bar_to_cgs, Temp_at_1bar, metalli)
#    print(f"The optical depth at 1 bar in our model: {optical_depth_our_case}")

    #Step 4: deduce which intrinsic Temperature is needed to recover this optical depth at this Pressure and Temperature
    T_int_lin = np.linspace(29,500,1000)
    Tint = find_Tint(optical_depth_our_case,T_int_lin, final_gamma, Temp_at_1bar, 50)
#    print(f"The intrinsic temperature {Tint} in our model")

    #Step 5: Using an upper and lower limit on temperature, find the max optical depth and minimal optical depth
    possible_depths = np.logspace(-10,10,1000)
    final_optical_depth  = find_tau(possible_depths,Tint, final_gamma, T_forpress, 1e5)
#    print(f"The optical depth until we go {final_optical_depth}")

    aze = pressure_finder_given_opacity(final_optical_depth, thermal_opacity, grav_at_1bar)/(10**6)
    return aze

def Find_atm_T(P, T, met, surf_grav):
    def test_find_atm_T(atmo_T):
        return Pressure_at_Temperature_From_1_bar(atmo_T, surf_grav, met,T) - P
    T_atmo_1bar = fsolve(test_find_atm_T, 1e2)
    return T_atmo_1bar[0]
def Pressure_and_Temperature_to_profile(P, T, met, surf_grav,grid_resolution):
    #Creates pressure and temperature profile given one pressure and temperature
    #optical depths 
    #T in Kelvin
    #met in [M/H], different then how we define elsewhere
    # the surface gravity
    #P in bar
    T_atmo_1bar = Find_atm_T(P, T, met, surf_grav)
    All_Temperatures, All_pressures, Tint, T_eff_end = Profile_From_1_bar(T_atmo_1bar, surf_grav, met, grid_resolution,T)
    return All_Temperatures, All_pressures, Tint, T_eff_end, T_atmo_1bar

In [12]:
def Find_T_from_atmo_and_Pressure(T_at_1bar, Pressure_at_layer, met, surf_grav):
    #Finds the Temperature when given atmospheric temperature at 1 bar and pressure
    #T_at_1bar, the temperature at the 1 bar level
    #The pressure at our starting layer,
    #met metallicity as in [M/H] 
    #surf_grav, surface gravity woop

    Press_1bar_to_cgs = 10**6 #to dyne/cm^2, this is Pressure of 1bar in dyne/cm^2
    
    #Step 1: deduce optical depth at 76K level, the Uranus we know of today 
    optical_depth_76K = optical_depth_finder(Press_1bar_to_cgs, 76, met,surf_grav)
#    print(f"Optical depth at 76K in Uranus as of today: {optical_depth_76K}")
    
    #Step 2: deduce an estimation for gamma 
    gammas = np.logspace(-2,0,1000)
    T_eff_final = 59.1 
    T_eq_final = 58.2
    T_int_final = (T_eff_final**(4) - T_eq_final**(4))**(1/4)
    final_gamma = find_gamma(optical_depth_76K,T_int_final, gammas, 76, 0.3)
#    print(f"The gamma to be used is : {final_gamma}")
    
    #Step 3: deduce the optical depth at which our 1bar-level is
    optical_depth_our_case = Press_1bar_to_cgs*opacity(Press_1bar_to_cgs, T_at_1bar, met)/surf_grav
    thermal_opacity = opacity(Press_1bar_to_cgs, T_at_1bar, met)
#    print(f"The optical depth at 1 bar in our model: {optical_depth_our_case}")

    #Step 4: deduce which intrinsic Temperature is needed to recover this optical depth at this Pressure and Temperature
    T_int_lin = np.linspace(29,500,1000)
    Tint = find_Tint(optical_depth_our_case,T_int_lin, final_gamma, T_at_1bar, 50)
#    print(f"The intrinsic temperature {Tint} in our model")

    #Step 5: Deduce the optical depth needed for our pressure
    pressure_tau = optical_depth_finder_given_opacity(Pressure_at_layer*10**6, thermal_opacity, surf_grav)
    #Step 6: Given everything now, deduce the temperature at our layer
    Final_T = Temp_profile(pressure_tau, Tint, final_gamma)
    return Final_T

def Create_Temperature_Constant_Pressure(Constant_Pressure, starting_Temperature_at_layer, ending_Temperature_1bar, met, surf_grav, timesteps,grid_resolution):
    #Here we start from a temperature with a pressure there. Next we construct how that pressure evolves in time, how it cools down, while also solving
    # temperature-pressure profiles at all different timesteps
    # Tatmo returns a list of 1 bar temperatures during the timesteps
    # Solidus returns wether at that timestep we still have melt or not 
    # Tlayer returns temperatures at those layers
    # Pressures_atmo, Temperatures_atmo return the pressure-temperature profile

    #Constant Pressure: The pressure found back at our layer in bar
    #starting_Temperature_at_layer: the temperature at our layer 
    #ending_Temperature_1bar: the ending temperature, taken often 76 K
    # met metallicity as in [M/H], often taken 1.5
    #surf_grav in cgs, often 887
    #timesteps, the amount of steps in time
    #gridresolution : how many values in our atmosphere

    #Step 1: from our pressure and layer, construct a temperature-pressure profile, from this we can also get pressure at 1 bar
    Temperature_profile0, Pressure_profile0, Tint0, T_eff0, T_atmo_1bar0 = Pressure_and_Temperature_to_profile(Constant_Pressure, starting_Temperature_at_layer, met, surf_grav,grid_resolution)
    
    #Step 2: create a linspace of temperatures from then to now
    T_atmo_lin = np.linspace(T_atmo_1bar0,ending_Temperature_1bar, timesteps)

    #Step 3: Find the layers temperatures for the different 1 bar temperatures
    Tlayer = []
    for Ts in T_atmo_lin:
        Tlayer.append(Find_T_from_atmo_and_Pressure(Ts, Constant_Pressure, met, surf_grav))
    Tlayer = np.array(Tlayer)

    #Step 4: Consctruct from the different layers different pressure temperature profiles, the final state, the state as of today often gives
    #numerical errors, so we exclude it first and then calculate
    T_profiles = np.zeros((timesteps, grid_resolution))
    P_profiles = np.zeros((timesteps, grid_resolution))
    Tint_profiles = np.zeros((timesteps, 1))
    T_eff_profiles = np.zeros((timesteps, 1))
    T_atmo_1bar_profiles = np.zeros((timesteps, 1))
    Melt_profiles = np.zeros((timesteps, 1))
    
    for i in range(len(Tlayer)-1):
        Ti, Pi, Tinti, T_effi, T_atmo_1bari = Pressure_and_Temperature_to_profile(Constant_Pressure, Tlayer[i], met, surf_grav,grid_resolution)
        T_profiles[i] = Ti
        P_profiles[i] = Pi
        Tint_profiles[i] = Tinti
        T_eff_profiles[i] = T_effi
        T_atmo_1bar_profiles[i] = T_atmo_1bari

    #Step 5 construct the final layer
    T_profiles[-1], P_profiles[-1], Tint_profiles[-1], T_eff_profiles[-1] = Profile_From_1_bar(ending_Temperature_1bar, surf_grav, met, grid_resolution,Tlayer[-1])
    T_atmo_1bar_profiles[-1] = ending_Temperature_1bar

    #Step 6, for our starting layers, conclude wether magma ocean or not with solidus
    Solidus_line = peridotite_solidus(Constant_Pressure)
    for i in range(len(Tlayer)):
        if Tlayer[i] < Solidus_line:
            Melt_profiles[i] = False
        else:
            Melt_profiles[i] = True
    return Tlayer, T_atmo_1bar_profiles,T_profiles, P_profiles, Tint_profiles, T_eff_profiles, Melt_profiles

In [13]:
def give_melt(Pressure, Temperature):
    #Give pressure in bar
    Tline = peridotite_solidus(Pressure)
    if Tline <= Temperature:
        return 1
    elif Tline> Temperature:
        return 0

def create_melt_list(excel_file):
    #Excel file from atmodeller to generate a list with values for 0 or 1
    mantle_melt = []
    Pressures = pd.read_excel(excel_file, sheet_name="state")["pressure"]
    Temperatures = pd.read_excel(excel_file, sheet_name="state")["temperature"]

    for i in range(len(Pressures)):
        mantle_melt.append(give_melt(Pressures[i], Temperatures[i]))


    return mantle_melt

def create_melt(Pressures, Temperatures):
    #Excel file from Atmodeller to generate a list with values for 0 or 1
    mantle_melt = []
    for i in range(len(Pressures)):
        mantle_melt.append(give_melt(Pressures[i], Temperatures[i]))


    return mantle_melt

In [14]:
file = f'./excel_Uranus_time/HOSi/rho3000/core_boundary/HOSi_magma_sol_real_1xsolar_melt100_fO2-4_T2477K.xlsx'
melti = create_melt_list(file)
print(melti)

[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [15]:
def create_phase_lookup(filename):
    """
    Create a simple phase lookup function
    """
    # Load data
    data = []
    with open(filename, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                vals = line.split()
                float(vals[0])  # Test if numeric
                data.append([float(x) for x in vals[:11]])
            except (ValueError, IndexError):
                continue
    
    data = np.array(data)
    
    # Phase names
    phase_names = {
        -1: "Ice Ih", -2: "Ice II", -3: "Ice III",
        -5: "Ice V", -6: "Ice VI", -7: "Ice VII",
        3: "Vapor", 4: "Liquid", 5: "Supercritical"
    }
    
    # Extract columns
    temps = data[:, 1]      # Temperature in K
    press = data[:, 0] / 1e9  # Pressure in GPa
    phases = data[:, 10]     # Phase ID
    
    def lookup(T, P):
        """Return phase name at T(K) and P(GPa)"""
        # Find closest point
        distances = (temps - T)**2 + (press - P)**2
        idx = np.argmin(distances)
        phase_id = int(phases[idx])
        return phase_names.get(phase_id, f"Phase {phase_id}")
    
    return lookup

filename = '/home/mathias/H2Oeq/AQUA/Tables/aqua_eos_pt_v1_0.dat'
get_phase = create_phase_lookup(filename)

In [16]:
def read_aqua_simple(filename):
    """
    Read AQUA data file, skipping header lines that start with text
    """
    data = []
    line_count = 0
    skipped_count = 0
    
    with open(filename, 'r') as f:
        for line in f:
            line = line.strip()
            line_count += 1
            
            # Skip empty lines
            if not line:
                continue
            
            # Try to convert to float - if it fails, skip the line
            try:
                # Split and try to convert first element to float
                vals = line.split()
                if vals:  # Check if line has content
                    float(vals[0])  # Test if first value is a number
                    # If successful, convert all values to float
                    data.append([float(x) for x in vals[:11]])
            except ValueError:
                # This line contains text - skip it
                skipped_count += 1
                continue
    
    print(f"Read {line_count} total lines")
    print(f"Skipped {skipped_count} header/comment lines")
    print(f"Loaded {len(data)} data points")
    
    if len(data) == 0:
        raise ValueError("No data points were loaded! Check file format.")
    
    # Convert to DataFrame
    df = pd.DataFrame(data, columns=[
        'press_Pa', 'temp_K', 'rho', 'ad_grad', 's', 'u', 
        'c', 'mmw', 'x_ion', 'x_d', 'phase_id'
    ])
    
    # Convert pressure to GPa for convenience
    df['press_GPa'] = df['press_Pa'] / 1e9
    
    # Ensure phase_id is integer
    df['phase_id'] = df['phase_id'].astype(int)
    
    # Phase names mapping
    phase_names = {
        -1: "Ice Ih",
        -2: "Ice II",
        -3: "Ice III",
        -5: "Ice V",
        -6: "Ice VI",
        -7: "Ice VII",
        -10: "Ice X",
        3: "Vapor",
        4: "Liquid",
        5: "Supercritical/Superionic"
    }
    
    def get_phase(T, P):
        """Get phase at temperature T(K) and pressure P(GPa)"""
        # Find closest point
        distances = np.sqrt(
            (df['temp_K'] - T)**2 + 
            (df['press_GPa'] - P)**2
        )
        idx = distances.idxmin()
        pid = df.loc[idx, 'phase_id']
        
        # Also return density and actual T,P for verification
        actual_T = df.loc[idx, 'temp_K']
        actual_P = df.loc[idx, 'press_GPa']
        density = df.loc[idx, 'rho']
        
        phase_name = phase_names.get(pid, f"Unknown Phase ({pid})")
        
        return {
            'phase': phase_name,
            'phase_id': pid,
            'density': density,
            'actual_T': actual_T,
            'actual_P': actual_P,
            'distance': distances[idx]
        }
    
    # Print summary
    print(f"\n✅ Successfully loaded {len(df):,} data points")
    print(f"   Temperature range: {df['temp_K'].min():.0f} - {df['temp_K'].max():.0f} K")
    print(f"   Pressure range: {df['press_GPa'].min():.6f} - {df['press_GPa'].max():.2f} GPa")
    print(f"   Phase IDs present: {sorted(df['phase_id'].unique())}")
    
    return df, get_phase

# Alternative: Using pandas with comment handling
def read_aqua_pandas(filename):
    """
    Alternative method using pandas with comment handling
    """
    # Try to find where the numeric data starts
    with open(filename, 'r') as f:
        for i, line in enumerate(f):
            try:
                # Try to parse first few numbers
                vals = line.strip().split()
                if vals:
                    float(vals[0])
                    float(vals[1])
                    # If successful, this is the start of data
                    skip_rows = i
                    print(f"Data starts at line {i+1}")
                    break
            except (ValueError, IndexError):
                continue
    
    # Read with pandas, skipping header rows
    df = pd.read_csv(
        filename,
        sep=r'\s+',
        skiprows=skip_rows,
        names=['press_Pa', 'temp_K', 'rho', 'ad_grad', 's', 'u', 
               'c', 'mmw', 'x_ion', 'x_d', 'phase_id'],
        engine='python'
    )
    
    df['press_GPa'] = df['press_Pa'] / 1e9
    df['phase_id'] = df['phase_id'].astype(int)
    
    return df

# ===== USAGE =====

filename = '/home/mathias/H2Oeq/AQUA/Tables/aqua_eos_pt_v1_0.dat'

# Method 1: Simple function
print("="*60)
print("LOADING AQUA DATA")
print("="*60)
df, get_phase = read_aqua_simple(filename)

# Test points
print("\n" + "="*60)
print("TEST POINTS")
print("="*60)

test_points = [
    (300, 0.0001),   # Near surface
    (500, 1),         # 1 GPa
    (1000, 10),       # 10 GPa
    (2000, 50),       # 50 GPa
    (4000, 100),      # 100 GPa
]

for T, P in test_points:
    result = get_phase(T, P)
    print(f"T={T:4.0f}K, P={P:6.2f}GPa → {result['phase']:20s} "
          f"(ρ={result['density']:8.1f} kg/m³) "
          f"[closest point: T={result['actual_T']:.0f}K, P={result['actual_P']:.3f}GPa]")

# Quick lookup function for single use
def quick_phase(T, P):
    """Quick one-line phase lookup"""
    return get_phase(T, P)['phase']


LOADING AQUA DATA
Read 329012 total lines
Skipped 19 header/comment lines
Loaded 328993 data points

✅ Successfully loaded 328,993 data points
   Temperature range: 100 - 100000 K
   Pressure range: 0.000000 - 400000.00 GPa
   Phase IDs present: [np.int64(-10), np.int64(-7), np.int64(-6), np.int64(-5), np.int64(-3), np.int64(-2), np.int64(-1), np.int64(3), np.int64(4), np.int64(5)]

TEST POINTS
T= 300K, P=  0.00GPa → Liquid               (ρ=   996.0 kg/m³) [closest point: T=302K, P=0.000GPa]
T= 500K, P=  1.00GPa → Liquid               (ρ=  1139.4 kg/m³) [closest point: T=501K, P=1.003GPa]
T=1000K, P= 10.00GPa → Supercritical/Superionic (ρ=  1618.3 kg/m³) [closest point: T=1000K, P=10.034GPa]
T=2000K, P= 50.00GPa → Supercritical/Superionic (ρ=  2338.4 kg/m³) [closest point: T=1995K, P=50.297GPa]
T=4000K, P=100.00GPa → Supercritical/Superionic (ρ=  2724.1 kg/m³) [closest point: T=3981K, P=100.366GPa]


In [17]:
def list_phases(Temperatures, Pressures):
    #pressures in Gpa, Temperatures in kelvin
    phases = []
    for i in range(len(Temperatures)):
        phase = quick_phase(Temperatures[i], Pressures[i])
        phases.append(phase)
        phases = list(set(phases))
    return phases

In [18]:
filename = f"/home/mathias/Margtest/Magrathea/result_final0/results.xlsx"

params = pd.read_excel(filename, sheet_name="Run_Parameters")
mask = params["mass_of_hydro"] == 0
Radii = pd.read_excel(filename, sheet_name="Results")["Radius (earth)"][mask]
Masses = pd.read_excel(filename, sheet_name="Results")["M (earth)"][mask]
Temperatures_layer = pd.read_excel(filename, sheet_name="Results")["T (K)"][mask]
Pressures_layer = pd.read_excel(filename, sheet_name="Results")["P (GPa)"][mask]

Radii = Radii.iloc[::-1].reset_index(drop=True)
Masses = Masses.iloc[::-1].reset_index(drop=True)
Temperatures_layer = Temperatures_layer.iloc[::-1].reset_index(drop=True)
Pressures_layer = Pressures_layer.iloc[::-1].reset_index(drop=True)
Pressures_layer = np.array(Pressures_layer)*1e4

Radii = Radii.iloc[125:].reset_index(drop=True)
Masses = Masses.iloc[125:].reset_index(drop=True)
Temperatures_layer = Temperatures_layer.iloc[125:].reset_index(drop=True)
Pressures_layer = Pressures_layer[125:]
print(Temperatures_layer)
print(Pressures_layer)

0      1880.29960
1      1873.67910
2      1866.96770
3      1860.17250
4      1853.28780
          ...    
121     677.22133
122     656.14561
123     635.23519
124     619.71295
125     607.76130
Name: T (K), Length: 126, dtype: float64
[ 921800.47          923062.3400000001  924313.0700000001
  925573.85          926813.14          928093.0700000001
  929388.78          930676.7100000001  932019.96
  933330.07          934690.5599999999  936032.0299999999
  937423.13          938825.83          940250.14
  941699.5           943161.02          944605.81
  946058.27          947423.3300000001  948831.8200000001
  950209.73          951489.4800000001  952798.43
  954114.73          955401.27          956688.2200000001
  957962.54          959236.42          960467.01
  961734.74          963009.4099999999  964234.24
  965431.88          966686.1           967951.29
  969245.49          970607.25          972020.5900000001
  973537.78          975104.48          976772.27
  978509.77  

In [24]:
#TESTTTT
#sidenote intrinsic temperature 
#going back from 1 bar towards history, see when it becomes unphysical 
init_metallicitys = [1]
init_metallicity = 1
mantle_melt_fraction= 1
model_magma_nosol_ideal = EquilibriumModel(species_HOSi_magma_nosol_ideal)
model_magma_sol_ideal = EquilibriumModel(species_HOSi_magma_sol_ideal)
model_magma_sol_real = EquilibriumModel(species_HOSi_magma_sol_real)
rho_centre = 3000
fO2_log10_shift = -3


for i in range(len(Radii)):
    print(Radii[i])
    print(Masses[i])
    print(Temperatures_layer[i])
    radius = Radii[i]*R_earth
    masses = Masses[i]*M_earth
    temp_layer = Temperatures_layer[i]
    shift = f"{fO2_log10_shift:.2g}"
    file_name = f"./excel_Uranus_time/HOSi/rho{rho_centre}/core_boundary/HOSi_magma_sol_real_{init_metallicity}xsolar_melt{round(mantle_melt_fraction * 100)}_fO2{shift}_T{temp_layer:.4g}K.xlsx"
    P = pd.read_excel(file_name, sheet_name="state")["pressure"]
    T = pd.read_excel(file_name, sheet_name="state")["temperature"]
    planet_masses = pd.read_excel(file_name, sheet_name="state")["planet_mass"]
    planet = Planet(
        temperature=temp_layer,
        planet_mass=masses,
        mantle_melt_fraction=1,
        surface_radius=radius,
        )
    #Based on layer pressure and magrathea
    h_kgs = hmps / 100 * M_uranus  # kg

    si_kgs_solar = h_kgs * Si_masspercent_lodders09 / H_masspercent_lodders09
    mg_kgs_solar = h_kgs * Mg_masspercent_lodders09 / H_masspercent_lodders09
    o_kgs_solar = h_kgs * O_masspercent_lodders09 / H_masspercent_lodders09
    c_kgs_solar = h_kgs * C_masspercent_lodders09 / H_masspercent_lodders09
    n_kgs_solar = h_kgs * N_masspercent_lodders09 / H_masspercent_lodders09
    he_kgs_solar = h_kgs * He_masspercent_lodders09 / H_masspercent_lodders09
    si_kg_magma = Si_massfraction_palme14 * masses
    for init_metallicity in init_metallicitys:
        mass_constraints = {
        "H": h_kgs,
        "Si": init_metallicity * si_kgs_solar + mantle_melt_fraction * si_kg_magma,
    }
        state_gas: ThermodynamicStateProtocol = ThermodynamicState(T, P,mass=5.04e25, melt_fraction=1)
    # Magma - No solubility - Ideal Gas
        model_magma_nosol_ideal.solve(
                state=state_gas,
                mass_constraints=mass_constraints,
                fugacity_constraints={O2_rg.name: IronWustiteBuffer(fO2_log10_shift)},
                solver_parameters=SolverParameters(multistart=20)
            )
        output_magma_nosol_ideal = model_magma_nosol_ideal.output
            # output_magma_nosol_ideal.quick_look()
        output_magma_nosol_ideal.to_excel(
            f"./excel_Uranus_time/HOSi/rho{rho_centre}/core_boundary/HOSi_magma_nosol_ideal_{init_metallicity}xsolar_melt{round(mantle_melt_fraction * 100)}_fO2{shift}_T{temp_layer:.4g}K_comparison2"
        )
        output_magma_nosol_ideal.to_pickle(
            f"./excel_Uranus_time/HOSi/rho{rho_centre}/core_boundary/HOSi_magma_nosol_ideal_{init_metallicity}xsolar_melt{round(mantle_melt_fraction * 100)}_fO2{shift}_T{temp_layer:.4g}K_comparison2"
        )
    
        # Magma - Soluility - Ideal Gas
        initial_log_number_moles = output_magma_nosol_ideal.log_number_moles
        model_magma_sol_ideal.solve(
            state=state_gas,
            mass_constraints=mass_constraints,
            initial_log_number_moles=initial_log_number_moles,
            fugacity_constraints={O2_rg.name: IronWustiteBuffer(fO2_log10_shift)},
            solver_parameters=SolverParameters(multistart=20)
        )
        output_magma_sol_ideal = model_magma_sol_ideal.output
        # output_magma_sol_ideal.quick_look()
        output_magma_sol_ideal.to_excel(
            f"./excel_Uranus_time/HOSi/rho{rho_centre}/core_boundary/HOSi_magma_sol_ideal_{init_metallicity}xsolar_melt{round(mantle_melt_fraction * 100)}_fO2{shift}_T{temp_layer:.4g}K_comparison2"
        )
        output_magma_sol_ideal.to_pickle(
            f"./excel_Uranus_time/HOSi/rho{rho_centre}/core_boundary/HOSi_magma_sol_ideal_{init_metallicity}xsolar_melt{round(mantle_melt_fraction * 100)}_fO2{shift}_T{temp_layer:.4g}K_comparison2"
        )
        # Magma - Solubility - Real Gas
        initial_log_number_moles = output_magma_sol_ideal.log_number_moles
        model_magma_sol_real.solve(
            state=state_gas,
            mass_constraints=mass_constraints,
            initial_log_number_moles=initial_log_number_moles,
            fugacity_constraints={O2_rg.name: IronWustiteBuffer(fO2_log10_shift)},
            solver_parameters=SolverParameters(multistart=20)
        )
        output_magma_sol_real = model_magma_sol_real.output
        output_magma_sol_real.quick_look()
        output_magma_sol_real.to_excel(
            f"./excel_Uranus_time/HOSi/rho{rho_centre}/core_boundary/HOSi_magma_sol_real_{init_metallicity}xsolar_melt{round(mantle_melt_fraction * 100)}_fO2{shift}_T{temp_layer:.4g}K_comparison2"
        )
        output_magma_sol_real.to_pickle(
            f"./excel_Uranus_time/HOSi/rho{rho_centre}/core_boundary/HOSi_magma_sol_real_{init_metallicity}xsolar_melt{round(mantle_melt_fraction * 100)}_fO2{shift}_T{temp_layer:.4g}K_comparison2"
        )

[11:50:49 - atmodeller.classes             - INFO     ] - species_network = ('H2O_g: IdealGas, NoSolubility', 'H2_g: IdealGas, NoSolubility', 'O2_g: IdealGas, NoSolubility', 'OSi_g: IdealGas, NoSolubility', 'H4Si_g: IdealGas, NoSolubility', 'O2Si_cd: CondensateActivity, NoSolubility')
[11:50:49 - atmodeller.classes             - INFO     ] - Thermodynamic data requires temperatures between 200 K and 6000 K
[11:50:49 - atmodeller.classes             - INFO     ] - reactions = {0: '2.0 H2O_g = 2.0 H2_g + 1.0 O2_g',
 1: '3.0 H2_g + 1.0 OSi_g = 1.0 H2O_g + 1.0 H4Si_g',
 2: '1.0 H2O_g + 1.0 OSi_g = 1.0 H2_g + 1.0 O2Si_cd'}
[11:50:49 - atmodeller.classes             - INFO     ] - species_network = ('H2O_g: IdealGas, SolubilityPowerLaw', 'H2_g: IdealGas, SolubilityPowerLawLog10', 'O2_g: IdealGas, NoSolubility', 'OSi_g: IdealGas, NoSolubility', 'H4Si_g: IdealGas, NoSolubility', 'O2Si_cd: CondensateActivity, NoSolubility')
[11:50:49 - atmodeller.classes             - INFO     ] - Thermodynamic

1.9789044
11.4337
1880.2996


[11:50:57 - atmodeller.classes             - INFO     ] - Solve (robust) complete: 50 (100.00%) successful model(s)
[11:50:57 - atmodeller.classes             - INFO     ] - Multistart summary: 50 (100.00%) models(s) required 1 attempt(s)
[11:50:57 - atmodeller.classes             - INFO     ] - Solver steps (max) = 28
[11:50:57 - atmodeller.output_core         - DEBUG    ] - Creating Output
[11:50:57 - atmodeller.output              - INFO     ] - Writing output to excel
[11:50:57 - atmodeller.output_core         - INFO     ] - Computing to_dataframes output
[11:50:57 - atmodeller.output_core         - INFO     ] - Computing asdict output
[11:50:58 - atmodeller.output_core         - DEBUG    ] - Found O2_g so back-computing log10 shift for fO2
[11:50:59 - atmodeller.output              - INFO     ] - Output written to excel_Uranus_time/HOSi/rho3000/core_boundary/HOSi_magma_nosol_ideal_1xsolar_melt100_fO2-3_T1880K_comparison2.xlsx
[11:50:59 - atmodeller.output              - INFO     ]

KeyboardInterrupt: 

In [31]:
init_metallicitys = [1]
init_metallicity = 1
mantle_melt_fraction= 1
model_magma_nosol_ideal = EquilibriumModel(species_HOSi_magma_nosol_ideal)
model_magma_sol_ideal = EquilibriumModel(species_HOSi_magma_sol_ideal)
model_magma_sol_real = EquilibriumModel(species_HOSi_magma_sol_real)
rho_centre = 3000
fO2_log10_shift = -4
hmps = np.logspace(1,1, num=1)  # wt% H
for i in range(len(Radii)):
    radius = Radii[i]*R_earth
    masses = Masses[i]*M_earth
    temp_layer = Temperatures_layer[i]
    pressure_layer = Pressures_layer[i]
    print(masses)
    print(temp_layer)
    print(pressure_layer)
    shift = f"{fO2_log10_shift:.2g}"
    #Based on layer pressure and magrathea
    h_kgs = hmps / 100 * M_uranus  # kg

    si_kgs_solar = h_kgs * Si_masspercent_lodders09 / H_masspercent_lodders09
    mg_kgs_solar = h_kgs * Mg_masspercent_lodders09 / H_masspercent_lodders09
    o_kgs_solar = h_kgs * O_masspercent_lodders09 / H_masspercent_lodders09
    c_kgs_solar = h_kgs * C_masspercent_lodders09 / H_masspercent_lodders09
    n_kgs_solar = h_kgs * N_masspercent_lodders09 / H_masspercent_lodders09
    he_kgs_solar = h_kgs * He_masspercent_lodders09 / H_masspercent_lodders09
    si_kg_magma = Si_massfraction_palme14 * masses
    for init_metallicity in init_metallicitys:
        mass_constraints = {
        "H": h_kgs,
        "Si": init_metallicity * si_kgs_solar + mantle_melt_fraction * si_kg_magma,
    }
        state_gas: ThermodynamicStateProtocol = ThermodynamicState(temp_layer, pressure_layer,mass=1e19, melt_fraction=give_melt(temp_layer,pressure_layer))
        # Magma - No solubility - Ideal Gas
        model_magma_nosol_ideal.solve(
                state=state_gas,
                mass_constraints=mass_constraints,
                fugacity_constraints={O2_rg.name: IronWustiteBuffer(fO2_log10_shift)},
                solver_parameters=SolverParameters(multistart=20)
            )
        output_magma_nosol_ideal = model_magma_nosol_ideal.output
            # output_magma_nosol_ideal.quick_look()
        output_magma_nosol_ideal.to_excel(
            f"./TEST"
        )
        output_magma_nosol_ideal.to_pickle(
            f"./TEST"
        )
    
        # Magma - Soluility - Ideal Gas
        initial_log_number_moles = output_magma_nosol_ideal.log_number_moles
        model_magma_sol_ideal.solve(
            state=state_gas,
            mass_constraints=mass_constraints,
            initial_log_number_moles=initial_log_number_moles,
            fugacity_constraints={O2_rg.name: IronWustiteBuffer(fO2_log10_shift)},
            solver_parameters=SolverParameters(multistart=20)
        )
        output_magma_sol_ideal = model_magma_sol_ideal.output
        # output_magma_sol_ideal.quick_look()
        output_magma_sol_ideal.to_excel(
            f"./TEST"
        )
        output_magma_sol_ideal.to_pickle(
            f"./TEST"
        )
        # Magma - Solubility - Real Gas
        initial_log_number_moles = output_magma_sol_ideal.log_number_moles
        model_magma_sol_real.solve(
            state=state_gas,
            mass_constraints=mass_constraints,
            initial_log_number_moles=initial_log_number_moles,
            fugacity_constraints={O2_rg.name: IronWustiteBuffer(fO2_log10_shift)},
            solver_parameters=SolverParameters(multistart=50)
        )
        output_magma_sol_real = model_magma_sol_real.output
        initial_log_number_moles = output_magma_sol_ideal.log_number_moles
        print(initial_log_number_moles*1.5)
        output_magma_sol_real.quick_look()
        output_magma_sol_real.to_excel(
            f"./TEST1"
        )
        output_magma_sol_real.to_pickle(
            f"./TEST1"
        )

        state_gas: ThermodynamicStateProtocol = ThermodynamicState(temp_layer, pressure_layer,mass=masses, melt_fraction=give_melt(temp_layer,pressure_layer))
        model_magma_sol_real.solve(
            state=state_gas,
            mass_constraints=mass_constraints,
            initial_log_number_moles=initial_log_number_moles*1.5,
            fugacity_constraints={O2_rg.name: IronWustiteBuffer(fO2_log10_shift)},
            solver_parameters=SolverParameters(multistart=200)
        )
        output_magma_sol_real = model_magma_sol_real.output
        initial_log_number_moles = output_magma_sol_ideal.log_number_moles
        print(initial_log_number_moles)
        output_magma_sol_real.quick_look()
        output_magma_sol_real.to_excel(
            f"./TEST1"
        )
        output_magma_sol_real.to_pickle(
            f"./TEST1"
        )

[16:13:01 - atmodeller.classes             - INFO     ] - species_network = ('H2O_g: IdealGas, NoSolubility', 'H2_g: IdealGas, NoSolubility', 'O2_g: IdealGas, NoSolubility', 'OSi_g: IdealGas, NoSolubility', 'H4Si_g: IdealGas, NoSolubility', 'O2Si_cd: CondensateActivity, NoSolubility')
[16:13:01 - atmodeller.classes             - INFO     ] - Thermodynamic data requires temperatures between 200 K and 6000 K
[16:13:02 - atmodeller.classes             - INFO     ] - reactions = {0: '2.0 H2O_g = 2.0 H2_g + 1.0 O2_g',
 1: '3.0 H2_g + 1.0 OSi_g = 1.0 H2O_g + 1.0 H4Si_g',
 2: '1.0 H2O_g + 1.0 OSi_g = 1.0 H2_g + 1.0 O2Si_cd'}
[16:13:02 - atmodeller.classes             - INFO     ] - species_network = ('H2O_g: IdealGas, SolubilityPowerLaw', 'H2_g: IdealGas, SolubilityPowerLawLog10', 'O2_g: IdealGas, NoSolubility', 'OSi_g: IdealGas, NoSolubility', 'H4Si_g: IdealGas, NoSolubility', 'O2Si_cd: CondensateActivity, NoSolubility')
[16:13:02 - atmodeller.classes             - INFO     ] - Thermodynamic

6.82820564e+25
1880.2996
921800.47


[16:13:29 - atmodeller.classes             - INFO     ] - Solve (robust) complete: 1 (100.00%) successful model(s)
[16:13:29 - atmodeller.classes             - INFO     ] - Multistart summary: 1 (100.00%) models(s) required 1 attempt(s)
[16:13:29 - atmodeller.classes             - INFO     ] - Solver steps (max) = 60
[16:13:29 - atmodeller.output_core         - DEBUG    ] - Creating Output
[16:13:29 - atmodeller.output              - INFO     ] - Writing output to excel
[16:13:29 - atmodeller.output_core         - INFO     ] - Computing to_dataframes output
[16:13:29 - atmodeller.output_core         - INFO     ] - Computing asdict output
[16:13:30 - atmodeller.output_core         - DEBUG    ] - Found O2_g so back-computing log10 shift for fO2
[16:13:30 - atmodeller.output              - INFO     ] - Output written to TEST.xlsx
[16:13:30 - atmodeller.output              - INFO     ] - Writing output to pickle
[16:13:30 - atmodeller.output_core         - DEBUG    ] - Returning cached to_

[[88.51111252093825 95.42972922655164 33.66453998260147 64.3405175014062
  81.36231569742256 91.69046649170991]]


[16:15:40 - atmodeller.output              - INFO     ] - Writing output to excel
[16:15:40 - atmodeller.output_core         - INFO     ] - Computing to_dataframes output
[16:15:40 - atmodeller.output_core         - INFO     ] - Computing asdict output
[16:15:41 - atmodeller.output_core         - DEBUG    ] - Found O2_g so back-computing log10 shift for fO2
[16:15:42 - atmodeller.output              - INFO     ] - Output written to TEST1.xlsx
[16:15:42 - atmodeller.output              - INFO     ] - Writing output to pickle
[16:15:42 - atmodeller.output_core         - DEBUG    ] - Returning cached to_dataframes output
[16:15:42 - atmodeller.output              - INFO     ] - Output written to TEST1.pkl
[16:17:25 - atmodeller.classes             - INFO     ] - Solve (robust) complete: 0 (0.00%) successful model(s)
[16:17:25 - atmodeller.classes             - WARNING  ] - 1 (100.00%) model(s) still failed
[16:17:25 - atmodeller.classes             - INFO     ] - Multistart summary: 1 (10

OverflowError: cannot convert float infinity to integer

In [49]:
#TESTTTT
#sidenote intrinsic temperature 
#going back from 1 bar towards history, see when it becomes unphysical 
init_metallicitys = [1]
init_metallicity = 1
mantle_melt_fraction= 1
model_magma_nosol_ideal = EquilibriumModel(species_HOSi_magma_nosol_ideal)
model_magma_sol_ideal = EquilibriumModel(species_HOSi_magma_sol_ideal)
model_magma_sol_real = EquilibriumModel(species_HOSi_magma_sol_real)
rho_centre = 3000
fO2_log10_shift = -3

for i in range(len(Radii)):
    print(Radii[i])
    print(Masses[i])
    print(Temperatures_layer[i])
    radius = Radii[i]*R_earth
    masses = Masses[i]*M_earth
    temp_layer = Temperatures_layer[i]
    shift = f"{fO2_log10_shift:.2g}"
    file_name = f"./excel_Uranus_time/HOSi/rho{rho_centre}/core_boundary/HOSi_magma_sol_real_{init_metallicity}xsolar_melt{round(mantle_melt_fraction * 100)}_fO2{shift}_T{temp_layer:.4g}K.xlsx"
    P = pd.read_excel(file_name, sheet_name="state")["pressure"]
    T = pd.read_excel(file_name, sheet_name="state")["temperature"]
    planet_masses = pd.read_excel(file_name, sheet_name="state")["planet_mass"]
    planet = Planet(
        temperature=temp_layer,
        planet_mass=masses,
        mantle_melt_fraction=1,
        surface_radius=radius,
        )
    #Based on layer pressure and magrathea
    h_kgs = hmps / 100 * M_uranus  # kg

    si_kgs_solar = h_kgs * Si_masspercent_lodders09 / H_masspercent_lodders09
    mg_kgs_solar = h_kgs * Mg_masspercent_lodders09 / H_masspercent_lodders09
    o_kgs_solar = h_kgs * O_masspercent_lodders09 / H_masspercent_lodders09
    c_kgs_solar = h_kgs * C_masspercent_lodders09 / H_masspercent_lodders09
    n_kgs_solar = h_kgs * N_masspercent_lodders09 / H_masspercent_lodders09
    he_kgs_solar = h_kgs * He_masspercent_lodders09 / H_masspercent_lodders09
    si_kg_magma = Si_massfraction_palme14 * masses
    for init_metallicity in init_metallicitys:
        mass_constraints = {
        "H": h_kgs,
        "Si": init_metallicity * si_kgs_solar + mantle_melt_fraction * si_kg_magma,
    }
        state_gas: ThermodynamicStateProtocol = ThermodynamicState(T, P,mass=5.04e25, melt_fraction=1)
    # Magma - No solubility - Ideal Gas
        model_magma_nosol_ideal.solve(
                state=state_gas,
                mass_constraints=mass_constraints,
                fugacity_constraints={O2_rg.name: IronWustiteBuffer(fO2_log10_shift)},
                solver_parameters=SolverParameters(multistart=20)
            )
        output_magma_nosol_ideal = model_magma_nosol_ideal.output
            # output_magma_nosol_ideal.quick_look()
        output_magma_nosol_ideal.to_excel(
            f"./excel_Uranus_time/HOSi/rho{rho_centre}/core_boundary/HOSi_magma_nosol_ideal_{init_metallicity}xsolar_melt{round(mantle_melt_fraction * 100)}_fO2{shift}_T{temp_layer:.4g}K_comparison3"
        )
        output_magma_nosol_ideal.to_pickle(
            f"./excel_Uranus_time/HOSi/rho{rho_centre}/core_boundary/HOSi_magma_nosol_ideal_{init_metallicity}xsolar_melt{round(mantle_melt_fraction * 100)}_fO2{shift}_T{temp_layer:.4g}K_comparison3"
        )
    
        # Magma - Soluility - Ideal Gas
        initial_log_number_moles = output_magma_nosol_ideal.log_number_moles
        model_magma_sol_ideal.solve(
            state=state_gas,
            mass_constraints=mass_constraints,
            initial_log_number_moles=initial_log_number_moles,
            fugacity_constraints={O2_rg.name: IronWustiteBuffer(fO2_log10_shift)},
            solver_parameters=SolverParameters(multistart=20)
        )
        output_magma_sol_ideal = model_magma_sol_ideal.output
        # output_magma_sol_ideal.quick_look()
        output_magma_sol_ideal.to_excel(
            f"./excel_Uranus_time/HOSi/rho{rho_centre}/core_boundary/HOSi_magma_sol_ideal_{init_metallicity}xsolar_melt{round(mantle_melt_fraction * 100)}_fO2{shift}_T{temp_layer:.4g}K_comparison3"
        )
        output_magma_sol_ideal.to_pickle(
            f"./excel_Uranus_time/HOSi/rho{rho_centre}/core_boundary/HOSi_magma_sol_ideal_{init_metallicity}xsolar_melt{round(mantle_melt_fraction * 100)}_fO2{shift}_T{temp_layer:.4g}K_comparison3"
        )
        # Magma - Solubility - Real Gas
        initial_log_number_moles = output_magma_sol_ideal.log_number_moles
        model_magma_sol_real.solve(
            state=state_gas,
            mass_constraints=mass_constraints,
            initial_log_number_moles=initial_log_number_moles,
            fugacity_constraints={O2_rg.name: IronWustiteBuffer(fO2_log10_shift)},
            solver_parameters=SolverParameters(multistart=20)
        )
        output_magma_sol_real = model_magma_sol_real.output
        output_magma_sol_real.quick_look()
        output_magma_sol_real.to_excel(
            f"./excel_Uranus_time/HOSi/rho{rho_centre}/core_boundary/HOSi_magma_sol_real_{init_metallicity}xsolar_melt{round(mantle_melt_fraction * 100)}_fO2{shift}_T{temp_layer:.4g}K_comparison3"
        )
        output_magma_sol_real.to_pickle(
            f"./excel_Uranus_time/HOSi/rho{rho_centre}/core_boundary/HOSi_magma_sol_real_{init_metallicity}xsolar_melt{round(mantle_melt_fraction * 100)}_fO2{shift}_T{temp_layer:.4g}K_comparison3"
        )

[12:19:26 - atmodeller.classes             - INFO     ] - species_network = ('H2O_g: IdealGas, NoSolubility', 'H2_g: IdealGas, NoSolubility', 'O2_g: IdealGas, NoSolubility', 'OSi_g: IdealGas, NoSolubility', 'H4Si_g: IdealGas, NoSolubility', 'O2Si_cd: CondensateActivity, NoSolubility')
[12:19:26 - atmodeller.classes             - INFO     ] - Thermodynamic data requires temperatures between 200 K and 6000 K
[12:19:26 - atmodeller.classes             - INFO     ] - reactions = {0: '2.0 H2O_g = 2.0 H2_g + 1.0 O2_g',
 1: '3.0 H2_g + 1.0 OSi_g = 1.0 H2O_g + 1.0 H4Si_g',
 2: '1.0 H2O_g + 1.0 OSi_g = 1.0 H2_g + 1.0 O2Si_cd'}
[12:19:26 - atmodeller.classes             - INFO     ] - species_network = ('H2O_g: IdealGas, SolubilityPowerLaw', 'H2_g: IdealGas, SolubilityPowerLawLog10', 'O2_g: IdealGas, NoSolubility', 'OSi_g: IdealGas, NoSolubility', 'H4Si_g: IdealGas, NoSolubility', 'O2Si_cd: CondensateActivity, NoSolubility')
[12:19:26 - atmodeller.classes             - INFO     ] - Thermodynamic

2.0187458
12.0207
2617.2794


[12:19:41 - atmodeller.classes             - INFO     ] - Solve (robust) complete: 50 (100.00%) successful model(s)
[12:19:41 - atmodeller.classes             - INFO     ] - Multistart summary: 50 (100.00%) models(s) required 1 attempt(s)
[12:19:41 - atmodeller.classes             - INFO     ] - Solver steps (max) = 29
[12:19:41 - atmodeller.output_core         - DEBUG    ] - Creating Output
[12:19:41 - atmodeller.output              - INFO     ] - Writing output to excel
[12:19:41 - atmodeller.output_core         - INFO     ] - Computing to_dataframes output
[12:19:41 - atmodeller.output_core         - INFO     ] - Computing asdict output
[12:19:41 - atmodeller.output_core         - DEBUG    ] - Found O2_g so back-computing log10 shift for fO2
[12:19:42 - atmodeller.output              - INFO     ] - Output written to excel_Uranus_time/HOSi/rho3000/core_boundary/HOSi_magma_nosol_ideal_1xsolar_melt100_fO2-3_T2617K_comparison2.xlsx
[12:19:42 - atmodeller.output              - INFO     ]

2.018482
12.0166
2611.5222


[12:21:36 - atmodeller.output              - INFO     ] - Writing output to excel
[12:21:36 - atmodeller.output_core         - INFO     ] - Computing to_dataframes output
[12:21:36 - atmodeller.output_core         - INFO     ] - Computing asdict output
[12:21:36 - atmodeller.output_core         - DEBUG    ] - Found O2_g so back-computing log10 shift for fO2
[12:21:37 - atmodeller.output              - INFO     ] - Output written to excel_Uranus_time/HOSi/rho3000/core_boundary/HOSi_magma_nosol_ideal_1xsolar_melt100_fO2-3_T2612K_comparison2.xlsx
[12:21:37 - atmodeller.output              - INFO     ] - Writing output to pickle
[12:21:37 - atmodeller.output_core         - DEBUG    ] - Returning cached to_dataframes output
[12:21:37 - atmodeller.output              - INFO     ] - Output written to excel_Uranus_time/HOSi/rho3000/core_boundary/HOSi_magma_nosol_ideal_1xsolar_melt100_fO2-3_T2612K_comparison2.pkl
[12:21:37 - atmodeller.classes             - INFO     ] - Solve (robust) complete:

2.0181988
12.0126
2605.7658


... (294 outputs left out)

[12:36:10 - atmodeller.output_core         - DEBUG    ] - Found O2_g so back-computing log10 shift for fO2
[12:36:11 - atmodeller.output              - INFO     ] - Output written to excel_Uranus_time/HOSi/rho3000/core_boundary/HOSi_magma_nosol_ideal_1xsolar_melt100_fO2-3_T1706K_comparison2.xlsx
[12:36:11 - atmodeller.output              - INFO     ] - Writing output to pickle
[12:36:11 - atmodeller.output_core         - DEBUG    ] - Returning cached to_dataframes output
[12:36:11 - atmodeller.output              - INFO     ] - Output written to excel_Uranus_time/HOSi/rho3000/core_boundary/HOSi_magma_nosol_ideal_1xsolar_melt100_fO2-3_T1706K_comparison2.pkl
[12:36:12 - atmodeller.classes             - INFO     ] - Solve (robust) complete: 24 (48.00%) successful model(s)
[12:36:12 - atmodeller.classes             - WARNING  ] - 26 (52.00%) model(s) still failed
[12:36:12 - atmodeller.classes             - INFO     ] - Multistart summary: 26 (52.00%) models(s) required 0 attempt(s)
[12:36

1.9688246
11.2832
1699.3102


[12:36:18 - atmodeller.output_core         - DEBUG    ] - Found O2_g so back-computing log10 shift for fO2
[12:36:19 - atmodeller.output              - INFO     ] - Output written to excel_Uranus_time/HOSi/rho3000/core_boundary/HOSi_magma_nosol_ideal_1xsolar_melt100_fO2-3_T1699K_comparison2.xlsx
[12:36:19 - atmodeller.output              - INFO     ] - Writing output to pickle
[12:36:19 - atmodeller.output_core         - DEBUG    ] - Returning cached to_dataframes output
[12:36:19 - atmodeller.output              - INFO     ] - Output written to excel_Uranus_time/HOSi/rho3000/core_boundary/HOSi_magma_nosol_ideal_1xsolar_melt100_fO2-3_T1699K_comparison2.pkl
[12:36:20 - atmodeller.classes             - INFO     ] - Solve (robust) complete: 21 (42.00%) successful model(s)
[12:36:20 - atmodeller.classes             - WARNING  ] - 29 (58.00%) model(s) still failed
[12:36:20 - atmodeller.classes             - INFO     ] - Multistart summary: 29 (58.00%) models(s) required 0 attempt(s)
[12:36

OverflowError: cannot convert float infinity to integer

In [50]:
init_metallicitys = [1]
init_metallicity = 1
mantle_melt_fraction= 1
model_magma_nosol_ideal = EquilibriumModel(species_HHeCNOSi_magma_nosol_ideal)
model_magma_sol_ideal = EquilibriumModel(species_HHeCNOSi_magma_sol_ideal)
model_magma_sol_real = EquilibriumModel(species_HHeCNOSi_magma_sol_real)
rho_centre = 3000
fO2_log10_shifts = [-3]
for i in range(len(Radii)):
    print(Radii[i])
    print(Masses[i])
    print(Temperatures_layer[i])
    radius = Radii[i]*R_earth
    masses = Masses[i]*M_earth
    temp_layer = Temperatures_layer[i]
    shift = f"{fO2_log10_shift:.2g}"
    file_name = f"./excel_Uranus_time/HHeCNOSi/rho{rho_centre}/core_boundary/HHeCNOSi_magma_sol_real_{init_metallicity}xsolar_melt{round(mantle_melt_fraction * 100)}_fO2{shift}_T{temp_layer:.4g}K.xlsx"
    P = pd.read_excel(file_name, sheet_name="state")["pressure"]
    T = pd.read_excel(file_name, sheet_name="state")["temperature"]
    planet = Planet(
        temperature=temp_layer,
        planet_mass=masses,
        mantle_melt_fraction=1,
        surface_radius=radius,
        )
        #Based on layer pressure and magrathea
    h_kgs = hmps / 100 * M_uranus  # kg

    si_kgs_solar = h_kgs * Si_masspercent_lodders09 / H_masspercent_lodders09
    mg_kgs_solar = h_kgs * Mg_masspercent_lodders09 / H_masspercent_lodders09
    o_kgs_solar = h_kgs * O_masspercent_lodders09 / H_masspercent_lodders09
    c_kgs_solar = h_kgs * C_masspercent_lodders09 / H_masspercent_lodders09
    n_kgs_solar = h_kgs * N_masspercent_lodders09 / H_masspercent_lodders09
    he_kgs_solar = h_kgs * He_masspercent_lodders09 / H_masspercent_lodders09
    si_kg_magma = Si_massfraction_palme14 * masses
    for init_metallicity in init_metallicitys:
        mass_constraints = {
            "H": h_kgs,
            "He": he_kgs_solar,
            "C": init_metallicity * c_kgs_solar,
            "N": init_metallicity * n_kgs_solar,
            "Si": init_metallicity * si_kgs_solar + mantle_melt_fraction * si_kg_magma,
            }
        state_gas: ThermodynamicStateProtocol = ThermodynamicState(T, P,mass=5.04e25, melt_fraction=1)
        # Magma - No solubility - Ideal Gas
        model_magma_nosol_ideal.solve(
            state=state_gas,
            mass_constraints=mass_constraints,
            fugacity_constraints={O2_rg.name: IronWustiteBuffer(fO2_log10_shift)},
            solver_parameters=SolverParameters(multistart=20)
        )
        output_magma_nosol_ideal = model_magma_nosol_ideal.output
            # output_magma_nosol_ideal.quick_look()
        output_magma_nosol_ideal.to_excel(
            f"./excel_Uranus_time/HHeCNOSi/rho{rho_centre}/core_boundary/HHeCNOSi_magma_nosol_ideal_{init_metallicity}xsolar_melt{round(mantle_melt_fraction * 100)}_fO2{shift}_T{temp_layer:.4g}K_comparison"
        )
        output_magma_nosol_ideal.to_pickle(
            f"./excel_Uranus_time/HHeCNOSi/rho{rho_centre}/core_boundary/HHeCNOSi_magma_nosol_ideal_{init_metallicity}xsolar_melt{round(mantle_melt_fraction * 100)}_fO2{shift}_T{temp_layer:.4g}K_comparison"
        )
    
        # Magma - Soluility - Ideal Gas
        initial_log_number_moles = output_magma_nosol_ideal.log_number_moles
        model_magma_sol_ideal.solve(
            state=state_gas,
            mass_constraints=mass_constraints,
            initial_log_number_moles=initial_log_number_moles,
            fugacity_constraints={O2_rg.name: IronWustiteBuffer(fO2_log10_shift)},
            solver_parameters=SolverParameters(multistart=20)
        )
        output_magma_sol_ideal = model_magma_sol_ideal.output
        # output_magma_sol_ideal.quick_look()
        output_magma_sol_ideal.to_excel(
            f"./excel_Uranus_time/HHeCNOSi/rho{rho_centre}/core_boundary/HHeCNOSi_magma_sol_ideal_{init_metallicity}xsolar_melt{round(mantle_melt_fraction * 100)}_fO2{shift}_T{temp_layer:.4g}K_comparison"
        )
        output_magma_sol_ideal.to_pickle(
            f"./excel_Uranus_time/HHeCNOSi/rho{rho_centre}/core_boundary/HHeCNOSi_magma_sol_ideal_{init_metallicity}xsolar_melt{round(mantle_melt_fraction * 100)}_fO2{shift}_T{temp_layer:.4g}K_comparison"
        )
        # Magma - Solubility - Real Gas
        initial_log_number_moles = output_magma_sol_ideal.log_number_moles
        model_magma_sol_real.solve(
            state=state_gas,
            mass_constraints=mass_constraints,
            initial_log_number_moles=initial_log_number_moles,
            fugacity_constraints={O2_rg.name: IronWustiteBuffer(fO2_log10_shift)},
            solver_parameters=SolverParameters(multistart=15)
        )
        output_magma_sol_real = model_magma_sol_real.output
        output_magma_sol_real.quick_look()
        output_magma_sol_real.to_excel(
            f"./excel_Uranus_time/HHeCNOSi/rho{rho_centre}/core_boundary/HHeCNOSi_magma_sol_real_{init_metallicity}xsolar_melt{round(mantle_melt_fraction * 100)}_fO2{shift}_T{temp_layer:.4g}K_comparison"
        )
        output_magma_sol_real.to_pickle(
            f"./excel_Uranus_time/HHeCNOSi/rho{rho_centre}/core_boundary/HHeCNOSi_magma_sol_real_{init_metallicity}xsolar_melt{round(mantle_melt_fraction * 100)}_fO2{shift}_T{temp_layer:.4g}K_comparison"
        )

[12:36:43 - atmodeller.classes             - INFO     ] - species_network = ('H2O_g: IdealGas, NoSolubility', 'H2_g: IdealGas, NoSolubility', 'O2_g: IdealGas, NoSolubility', 'OSi_g: IdealGas, NoSolubility', 'H4Si_g: IdealGas, NoSolubility', 'O2Si_cd: CondensateActivity, NoSolubility', 'CO2_g: IdealGas, NoSolubility', 'CO_g: IdealGas, NoSolubility', 'CH4_g: IdealGas, NoSolubility', 'N2_g: IdealGas, NoSolubility', 'H3N_g: IdealGas, NoSolubility', 'He_g: IdealGas, NoSolubility')
[12:36:43 - atmodeller.classes             - INFO     ] - Thermodynamic data requires temperatures between 200 K and 6000 K
[12:36:43 - atmodeller.classes             - INFO     ] - reactions = {0: '2.0 H2_g + 0.5 O2Si_cd = 1.0 H2O_g + 0.5 H4Si_g',
 1: '0.5 H4Si_g + 1.0 CO2_g = 1.0 H2_g + 0.5 O2Si_cd + 1.0 CO_g',
 2: '1.0 H4Si_g + 1.0 CO2_g = 1.0 O2Si_cd + 1.0 CH4_g',
 3: '0.5 H4Si_g + 0.5 O2Si_cd = 1.0 H2_g + 1.0 OSi_g',
 4: '1.5 H2_g + 0.5 N2_g = 1.0 H3N_g',
 5: '2.0 H2_g + 1.0 O2Si_cd = 1.0 O2_g + 1.0 H4Si_g'}


2.0187458
12.0207
2617.2794


[12:37:24 - atmodeller.classes             - INFO     ] - Solve (robust) complete: 50 (100.00%) successful model(s)
[12:37:24 - atmodeller.classes             - INFO     ] - Multistart summary: 50 (100.00%) models(s) required 1 attempt(s)
[12:37:24 - atmodeller.classes             - INFO     ] - Solver steps (max) = 39
[12:37:24 - atmodeller.output_core         - DEBUG    ] - Creating Output
[12:37:24 - atmodeller.output              - INFO     ] - Writing output to excel
[12:37:24 - atmodeller.output_core         - INFO     ] - Computing to_dataframes output
[12:37:24 - atmodeller.output_core         - INFO     ] - Computing asdict output
[12:37:25 - atmodeller.output_core         - DEBUG    ] - Found O2_g so back-computing log10 shift for fO2
[12:37:27 - atmodeller.output              - INFO     ] - Output written to excel_Uranus_time/HHeCNOSi/rho3000/core_boundary/HHeCNOSi_magma_nosol_ideal_1xsolar_melt100_fO2-3_T2617K_comparison.xlsx
[12:37:27 - atmodeller.output              - INF

2.018482
12.0166
2611.5222


[12:44:30 - atmodeller.classes             - INFO     ] - Solve (robust) complete: 50 (100.00%) successful model(s)
[12:44:30 - atmodeller.classes             - INFO     ] - Multistart summary: 50 (100.00%) models(s) required 1 attempt(s)
[12:44:30 - atmodeller.classes             - INFO     ] - Solver steps (max) = 39
[12:44:30 - atmodeller.output_core         - DEBUG    ] - Creating Output
[12:44:31 - atmodeller.output              - INFO     ] - Writing output to excel
[12:44:31 - atmodeller.output_core         - INFO     ] - Computing to_dataframes output
[12:44:31 - atmodeller.output_core         - INFO     ] - Computing asdict output
[12:44:32 - atmodeller.output_core         - DEBUG    ] - Found O2_g so back-computing log10 shift for fO2
[12:44:34 - atmodeller.output              - INFO     ] - Output written to excel_Uranus_time/HHeCNOSi/rho3000/core_boundary/HHeCNOSi_magma_nosol_ideal_1xsolar_melt100_fO2-3_T2612K_comparison.xlsx
[12:44:34 - atmodeller.output              - INF

2.0181988
12.0126
2605.7658


... (28 outputs left out)

[12:49:14 - atmodeller.output_core         - DEBUG    ] - Found O2_g so back-computing log10 shift for fO2
[12:49:15 - atmodeller.output              - INFO     ] - Output written to excel_Uranus_time/HHeCNOSi/rho3000/core_boundary/HHeCNOSi_magma_nosol_ideal_1xsolar_melt100_fO2-3_T2520K_comparison.xlsx
[12:49:15 - atmodeller.output              - INFO     ] - Writing output to pickle
[12:49:15 - atmodeller.output_core         - DEBUG    ] - Returning cached to_dataframes output
[12:49:15 - atmodeller.output              - INFO     ] - Output written to excel_Uranus_time/HHeCNOSi/rho3000/core_boundary/HHeCNOSi_magma_nosol_ideal_1xsolar_melt100_fO2-3_T2520K_comparison.pkl
[12:49:15 - atmodeller.classes             - INFO     ] - Solve (robust) complete: 50 (100.00%) successful model(s)
[12:49:15 - atmodeller.classes             - INFO     ] - Multistart summary: 50 (100.00%) models(s) required 1 attempt(s)
[12:49:15 - atmodeller.classes             - INFO     ] - Solver steps (max) = 6
[

2.0137217
11.9485
2513.9911


[12:49:27 - atmodeller.output_core         - DEBUG    ] - Found O2_g so back-computing log10 shift for fO2
[12:49:29 - atmodeller.output              - INFO     ] - Output written to excel_Uranus_time/HHeCNOSi/rho3000/core_boundary/HHeCNOSi_magma_nosol_ideal_1xsolar_melt100_fO2-3_T2514K_comparison.xlsx
[12:49:29 - atmodeller.output              - INFO     ] - Writing output to pickle
[12:49:29 - atmodeller.output_core         - DEBUG    ] - Returning cached to_dataframes output
[12:49:29 - atmodeller.output              - INFO     ] - Output written to excel_Uranus_time/HHeCNOSi/rho3000/core_boundary/HHeCNOSi_magma_nosol_ideal_1xsolar_melt100_fO2-3_T2514K_comparison.pkl
[12:49:29 - atmodeller.classes             - INFO     ] - Solve (robust) complete: 50 (100.00%) successful model(s)
[12:49:29 - atmodeller.classes             - INFO     ] - Multistart summary: 50 (100.00%) models(s) required 1 attempt(s)
[12:49:29 - atmodeller.classes             - INFO     ] - Solver steps (max) = 6
[

KeyboardInterrupt: 